In [2]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

backbone_model_id = "Qwen/Qwen3-4B"
peft_id = '/home/lxm/workspace/Shadow/outputs/mmlu_suite_lora_4B/mmlu_mmlu_full/lora'
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(backbone_model_id).to('cuda:7')

model = PeftModel.from_pretrained(
    base_model, peft_id, is_trainable=False).to(base_model.device)

model = model.eval()



Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.36it/s]


In [3]:
import torch

# Example GSM8K problem
# question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
question = "1+1=?"
question = "Ariel was playing basketball. 1 of her shots went in the hoop. 2 of her shots did not go in the hoop. How many shots were there in total?"
question = 'our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?'
question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'

user_content = f"Question: {question}"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？
assistant
<think>

</think>

我们来解决这个问题：

题目说彩珠按“2红3绿4黄”的顺序依次排列。也就是说，每组珠子的顺序是：

2个红色 → 3个绿色 → 4个黄色 → 重复这个循环。

因此，每组共有 2 + 3 + 4 = 9 颗珠子。

我们需要找出第509颗珠子的颜色。

### 第一步：确定509颗珠子中有多少个完整的循环

用509除以9，得到商和余数：

509 ÷ 9 = 56 余 5

这说明有56个完整的循环（每组9颗），再加上余数为5的第57组中的第5颗珠子。

### 第二步：确定第57组中的第5颗珠子的颜色

每组的顺序是：2红、3绿、4黄。

我们可以把顺序拆开：

- 前2个是红色
- 接下来的3个是绿色
- 剩下的4个是黄色

第57组中的前2个是红色，第3、4、5个是绿色，第6、7、8


In [5]:
model.push_to_hub("erin99/Qwen3-4B-MMLU-LoRA", private=True)

Processing Files (1 / 1): 100%|██████████| 94.4MB / 94.4MB, 7.38MB/s  
New Data Upload: 100%|██████████| 94.4MB / 94.4MB, 7.38MB/s  


CommitInfo(commit_url='https://huggingface.co/erin99/Qwen3-4B-MMLU-LoRA/commit/3261027bd8eb5ee450e43caf53f546151bd01236', commit_message='Upload model', commit_description='', oid='3261027bd8eb5ee450e43caf53f546151bd01236', pr_url=None, repo_url=RepoUrl('https://huggingface.co/erin99/Qwen3-4B-MMLU-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='erin99/Qwen3-4B-MMLU-LoRA'), pr_revision=None, pr_num=None)